# Lesson 06 — The Full Pipeline

In Lessons 03–05 we ran each step manually. Now we connect them with **job dependencies** — Batch ensures Job 2 only starts after Job 1 succeeds.

```
Upload video → [Job 1: extract frames] → [Job 2: embed frames] → Embeddings in S3
                        ↑ must SUCCEED before Job 2 even starts
```

This is the foundation of every ML batch pipeline.

## How Batch job dependencies work

```python
job1 = batch.submit_job(jobName="extract", ...)          # submitted now

job2 = batch.submit_job(
    jobName  = "embed",
    dependsOn= [{"jobId": job1["jobId"], "type": "N_TO_N"}],  # waits for job1
    ...
)
```

Both jobs are submitted instantly. Job 2 sits in `PENDING` state until Job 1 reaches `SUCCEEDED`. If Job 1 fails, Job 2 is automatically cancelled — **you never get stale embeddings from failed frames**.

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
print(f"Bucket: {os.environ['S3_BUCKET']}")
print(f"Queue : {os.environ['BATCH_JOB_QUEUE']}")

## Step 2 — Run the full pipeline

`run_pipeline.py` uploads the video, submits both jobs with dependencies, and polls until done.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "run_pipeline.py", "--video", "assets/sample.mp4"],
)
print("Exit code:", result.returncode)

## Step 3 — Verify: check that both outputs exist in S3

In [ ]:
import boto3, os

s3     = boto3.client("s3")
bucket = os.environ["S3_BUCKET"]

checks = [
    ("Frames",          "frames/sample/"),
    ("Embeddings",      "embeddings/sample/embeddings.npy"),
    ("Frame keys",      "embeddings/sample/frame_keys.json"),
]

for label, key_or_prefix in checks:
    try:
        if key_or_prefix.endswith("/"):
            resp  = s3.list_objects_v2(Bucket=bucket, Prefix=key_or_prefix, MaxKeys=1)
            found = resp.get("KeyCount", 0) > 0
        else:
            s3.head_object(Bucket=bucket, Key=key_or_prefix)
            found = True
    except Exception:
        found = False

    status = "✅" if found else "❌ not found"
    print(f"{label:15s}: {status}")

## Key Takeaway

> **`dependsOn` = declarative ordering.** You describe the dependencies; Batch runs the graph.  
> For production pipelines (10+ steps, branching, retries), tools like Apache Airflow or AWS Step Functions build on this same concept but add monitoring and retry logic.

---

## Next lesson → [07 — Scale & Cost](../07-scale-and-cost/notebook.ipynb)

We'll process multiple videos in parallel using Batch **array jobs**, and calculate the real cost.